# Week 3, Day 4: Ask an open model about WrestleMania

**Lesson:** [the Colab notebook](https://colab.research.google.com/drive/1hhR9Z-yiqjUe7pJjVQw4c74z_V3VchLy) linked from [week3/day4.ipynb](../../../week3/day4.ipynb), *Models*. **About 40 minutes.**

**What you practice here**

- Setting up 4-bit quantization with a `BitsAndBytesConfig`, and loading a model with it through `AutoModelForCausalLM`.
- Turning messages into token ids on the GPU, with the tokenizer's chat template and its generation prompt.
- Generating with the model's `generate`, streaming with a `TextStreamer`, and decoding only the reply's tokens.

**How this notebook works.** Run the first code cell once. Then, for each part below, write it in its cell, run that cell, and run the check under it. The checks hand your code stand-ins shaped like the real client and libraries: they record what your code sends and answer with canned replies, so nothing is called and nothing is spent. Write from memory, and look back at the lesson only when you are stuck.

When every check passes, save the notebook, run the checker and push, as in every session. The tracker then marks the lesson done.

In [ ]:
# Run this cell first. It finds plan/check.py in your fork and loads this exercise's checks.
import os, pathlib, sys
ROOT = next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "plan" / "check.py").is_file()), None)
if ROOT is None and ("google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ):
    # in Colab: fetch the checker from your fork on GitHub, and the packages Colab lacks
    import subprocess, urllib.request
    ROOT = pathlib.Path("/content/fork")
    (ROOT / "plan").mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve("https://raw.githubusercontent.com/starlest/llm_engineering/main/plan/check.py", ROOT / "plan" / "check.py")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "openai", "python-dotenv", "tiktoken"], check=False)
    print("In Colab, the checks run here too. To keep your work, use File > Save a copy in GitHub, to this notebook's path in your fork.")
if ROOT:
    sys.path.insert(0, str(ROOT / "plan"))
    try:
        from check import exercise
    except ImportError:
        raise ImportError("your plan/check.py is older than this notebook: run git pull, or copy the new one from the tracker page") from None
    check = exercise("week3/day4", globals())
else:  # outside your fork: write and try the parts here, and run the checks in your codespace
    def check(part=None):
        print("The checks are off here, as plan/check.py is not in a folder above this notebook. Run them in your codespace.")
    print("plan/check.py was not found above this notebook, so the checks are off here.")

## The task

The lesson goes below the pipelines to the models themselves. It loads Llama 3.2 in 4 bits, turns a chat into token
ids with the tokenizer, generates, and decodes the answer; then it wraps all that in a `generate` function that streams.
Do the same to ask an open model about WWE history.

transformers and torch are not in the checks' Python, so your functions take the class, the tokenizer or the model
they use as a parameter, and call it as the lesson does. The checks hand them stand-ins: a strict
`BitsAndBytesConfig`, a loader whose models report their memory, and a model whose generation answers with a canned
reply.

## What each call should give

```python
quant = four_bit_config(BitsAndBytesConfig, torch.bfloat16)
# the lesson's 4-bit config: double quantization, the nf4 type, computing in bfloat16

model = load_model(AutoModelForCausalLM, quant_config=quant)
# loads meta-llama/Llama-3.2-1B-Instruct onto the GPU, quantized with quant

tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-1B-Instruct")
inputs = chat_inputs(tokenizer, [{"role": "user", "content": "Who won the main event of WrestleMania III?"}])
# tensor([[128000, 128006, 9125, 128007, ...]], device='cuda:0'): one row of token ids, on the GPU

generate_reply(model, tokenizer, inputs, streamer_class=TextStreamer)
# prints the text as the model writes it, then returns the reply alone: 'Hulk Hogan won, pinning André the Giant ...'
```

In [ ]:
# The imports for this exercise. Add more here if you need them: the checks have the standard library, openai, pydantic, dotenv, numpy, bs4 and tiktoken.

### 1 · `four_bit_config(config_class, compute_dtype)`

Makes the lesson's 4-bit quantization config. `config_class` is `BitsAndBytesConfig`, and `compute_dtype` is a torch
dtype such as `torch.bfloat16`: both are passed in because the checks have neither transformers nor torch. Return a
`config_class` made with four settings, all by name: load in 4 bits, use double quantization, compute in
`compute_dtype`, and use the `"nf4"` quantization type. Spell the settings exactly: the real class ignores a setting
it does not know, which would load the full model, and the checks' stand-in refuses it.

<details><summary>Hint</summary>

```python
config_class(load_in_4bit=True, bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=compute_dtype,
             bnb_4bit_quant_type="nf4")
```

</details>

In [ ]:
def four_bit_config(config_class, compute_dtype):
    # Write four_bit_config here, as described above.
    raise NotImplementedError("four_bit_config is not written yet")

In [ ]:
check("four_bit_config")

### 2 · `load_model(auto_model, model="meta-llama/Llama-3.2-1B-Instruct", quant_config=None)`

Loads a causal language model onto the GPU, quantized when you give it a config. `auto_model` is
`AutoModelForCausalLM`. Load `model` once with its loading method, with `device_map="auto"` so that transformers
places it on the GPU, as the lesson does, and with `quant_config` as its quantization config. Pass the settings by
name, and return the model. With `quant_config=None` the model loads unquantized, and you may leave the quantization
config out.

<details><summary>Hint</summary>

```python
auto_model.from_pretrained(model, device_map="auto", quantization_config=quant_config)
```

</details>

In [ ]:
def load_model(auto_model, model="meta-llama/Llama-3.2-1B-Instruct", quant_config=None):
    # Write load_model here, as described above.
    raise NotImplementedError("load_model is not written yet")

In [ ]:
check("load_model")

### 3 · `chat_inputs(tokenizer, messages, device="cuda")`

Turns a chat into the token ids the model reads, on the GPU. First make the tokenizer's pad token its end-of-sequence
token, as the lesson does. Then apply the tokenizer's chat template to `messages`, asking for a PyTorch tensor of ids
and for the generation prompt: it ends the ids with the cue for the assistant's turn, so that the model answers
instead of carrying on the user's text. Move the tensor to `device` and return the moved one: moving a tensor returns
a new one and leaves the one you had where it was. The result is a batch holding one row of token ids.

<details><summary>Hint</summary>

```python
tokenizer.pad_token = tokenizer.eos_token
inputs = tokenizer.apply_chat_template(messages, return_tensors="pt", add_generation_prompt=True)
return inputs.to(device)
```

</details>

In [ ]:
def chat_inputs(tokenizer, messages, device="cuda"):
    # Write chat_inputs here, as described above.
    raise NotImplementedError("chat_inputs is not written yet")

In [ ]:
check("chat_inputs")

### 4 · `generate_reply(model, tokenizer, inputs, max_new_tokens=80, streamer_class=None)`

Generates the model's reply, streaming it if you ask, and returns the reply alone. When `streamer_class` is given, such
as `TextStreamer`, make a streamer from it with the tokenizer, and pass that streamer to the generation; when it is
`None`, pass no streamer. Generate once from `inputs`, with at most `max_new_tokens` new tokens. The first row of the
output holds the prompt's ids, then the new ones: decode only the new ones, skipping special tokens such as the
`<|eot_id|>` that ends them, and return that text.

<details><summary>Hint</summary>

```python
streamer = streamer_class(tokenizer) if streamer_class else None
outputs = model.generate(inputs, max_new_tokens=max_new_tokens, streamer=streamer)
return tokenizer.decode(outputs[0][len(inputs[0]):], skip_special_tokens=True)
```

</details>

In [ ]:
def generate_reply(model, tokenizer, inputs, max_new_tokens=80, streamer_class=None):
    # Write generate_reply here, as described above.
    raise NotImplementedError("generate_reply is not written yet")

In [ ]:
check("generate_reply")

## Check everything

The cell below runs all 8 checks again. When they all pass, save this notebook (Ctrl+S, or Cmd+S on a Mac), then run the checker and push, as in every session:

```bash
uv run python plan/check.py
git add -A
git commit -m "Week 3, Day 4 exercise"
git push
```

The checker runs the same checks on the saved notebook, and the tracker marks the lesson done. It reads only your imports, functions, classes and constants, so the experiments in your other cells never get in the way.

In [ ]:
check()

## Try it for real

Now run what you wrote on the real thing. This needs a GPU, so it runs in Google Colab: push this notebook, then [open it in Colab](https://colab.research.google.com/github/starlest/llm_engineering/blob/main/plan/exercises/week3/day4.ipynb), choose a T4 runtime, and run the cells from the top. The first cell fetches the checker from your fork, so the checks run in Colab too. Write the steps in the 2 cells at the end.

Colab works on its own copy of the notebook. To keep the Try code you write there, use **File > Save a copy in GitHub** (to this notebook's path in your fork), then run `git pull` in your codespace before you commit there again. Your parts are already in your fork, which is what the tracker reads.

Colab's free T4 is enough: Llama 3.2 1B in 4 bits takes about 1 GB of its memory. Both models are gated: with the
account your token belongs to, accept the Llama 3.2 terms on its Hugging Face page, and Gemma's terms at
https://huggingface.co/google/gemma-3-270m-it for step 4. Add your token to Colab's secrets as `HF_TOKEN`.

1. Keep the `!pip install` line in the first Try cell, and log in to Hugging Face with your `HF_TOKEN` secret.
2. Load Llama 3.2 1B in 4 bits: its tokenizer with `AutoTokenizer.from_pretrained`, then the model with `load_model`
   and `four_bit_config(BitsAndBytesConfig, torch.bfloat16)`. Print `model.get_memory_footprint() / 1e6`, its size in
   MB, and print `model` to find the embedding, the decoder layers and the LM head.
3. Ask it a WWE history question: `chat_inputs`, then `generate_reply(..., streamer_class=TextStreamer)`, and print
   the reply.
4. Free the GPU with `del`, `gc.collect()` and `torch.cuda.empty_cache()`, then ask `google/gemma-3-270m-it` the same
   question without quantization, and compare its footprint and its answer.

<details><summary>Stuck? One way to write it</summary>

```python
!pip install -q --upgrade bitsandbytes accelerate transformers==4.57.6
import torch
from google.colab import userdata
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TextStreamer

login(userdata.get("HF_TOKEN"))
LLAMA = "meta-llama/Llama-3.2-1B-Instruct"
QUESTION = [{"role": "system", "content": "You answer questions about WWE history in two sentences."},
            {"role": "user", "content": "Who won the main event of WrestleMania III, and why is it remembered?"}]

tokenizer = AutoTokenizer.from_pretrained(LLAMA)
model = load_model(AutoModelForCausalLM, LLAMA, four_bit_config(BitsAndBytesConfig, torch.bfloat16))
print(f"Memory footprint: {model.get_memory_footprint() / 1e6:,.1f} MB")
print(model)
inputs = chat_inputs(tokenizer, QUESTION)
print(generate_reply(model, tokenizer, inputs, streamer_class=TextStreamer))
```

```python
import gc

del model, inputs, tokenizer
gc.collect()
torch.cuda.empty_cache()

GEMMA = "google/gemma-3-270m-it"
tokenizer = AutoTokenizer.from_pretrained(GEMMA)
model = load_model(AutoModelForCausalLM, GEMMA)
print(f"Memory footprint: {model.get_memory_footprint() / 1e6:,.1f} MB")
print(generate_reply(model, tokenizer, chat_inputs(tokenizer, QUESTION)))
```

</details>

In [ ]:
!pip install -q --upgrade bitsandbytes accelerate transformers==4.57.6
# Steps 1 to 3: log in, load Llama 3.2 1B in 4 bits, and ask it about WWE history

In [ ]:
# Step 4: free the GPU, then try Gemma 3 270M without quantization